# Step 00. Every correction to the data, once, before anything else

**Source.** SomaScan v4.1 plasma proteomics in systemic lupus erythematosus. Imperial College
London (Leung, Pickering, Botto, Peters).

> https://doi.org/10.5281/zenodo.20342569

Download the record and unzip it so the files sit at `data/SLE_doi.10.5281_zenodo_20342569/`.

**Where this notebook's output goes.** The cohort matrices it writes are the one thing in the
pipeline that is *not* regenerable at will, redrawing them would move every downstream number, so they are written to `cohorts/` and committed. Everything a later step produces is written
to `data/run_artifacts/`, which is gitignored and safe to delete. `data/` holds the study
download and is gitignored too; no study data is committed. `../src/paths.R` is the only place
those locations are written down.

**The principle this notebook enforces.** Every fix to the data, log2, batch correction, protein
naming, de-duplication, happens once, across all samples, before any biological selection and
before any split. The earlier version of this step filtered to systemic lupus erythematosus (SLE) first and then corrected, which
fitted the batch model on a subset chosen by outcome and made the healthy volunteers impossible to
place on the same scale. Correction is a property of the assay, not of the question.

In [1]:
source("../src/paths.R")          # ROOT, RAW, COHORTS, ARTIFACTS -- see src/paths.R
stopifnot(dir.exists(RAW))

meta <- read.csv(raw("sample-metadata.csv"), check.names = FALSE)
c(rows = nrow(meta), SLE = sum(meta$Group == "SLE"), HV = sum(meta$Group == "HV"))

rows  SLE   HV 
 369  283   86

## One filter, and it is not biological

`Included_in_study == "Included"` drops 13 samples the study itself excluded. Healthy volunteers
stay in, because the batch correction below has to see them.

In [2]:
meta <- meta[meta$Included_in_study == "Included", ]
rownames(meta) <- meta$SampleId
table(Group = meta$Group, Batch = meta$Batch)

     Batch
Group   A   B
  HV   45  41
  SLE 217  53

## log2, then ComBat across all 356

**log2.** RFU intensities span about five orders of magnitude. On the raw scale a correlation is
dominated by whichever proteins are abundant.

**ComBat on `Batch`, across every sample, `mod = NULL`.** The correction is told nothing about
health status, it is unsupervised, like everything downstream of it.

**This is conservative,, and the cost has to be carried forward.** Healthy volunteers
are 17% of batch A and 44% of batch B, so part of the genuine batch-to-batch mean shift really is
composition rather than assay drift. With `mod = NULL`, ComBat absorbs some of the disease signal
into the correction. The alternative, `mod = model.matrix(~ Group)`, protects the group difference, but then any later unsupervised result separating healthy from SLE is partly an artifact of having
protected it. Under-detecting is the right direction to err for discovery.

**So: the healthy-vs-SLE separation reported in step 08 is a lower bound**, and must be described
that way wherever it appears.

In [3]:
suppressMessages(library(sva))

X <- read.csv(raw("abundance.csv"), row.names = 1, check.names = FALSE)
X <- as.matrix(X[rownames(meta), , drop = FALSE])
stopifnot(identical(rownames(X), rownames(meta)))

L <- t(ComBat(dat = t(log2(X)), batch = meta$Batch, mod = NULL,
              par.prior = TRUE, prior.plots = FALSE))
dim(L)

Found2batches



Adjusting for0covariate(s) or covariate level(s)



Standardizing Data across genes



Fitting L/S model and finding priors



Finding parametric adjustments



Adjusting the Data




[1]  356 7288

## Readable protein names

The abundance matrix is keyed by SOMAmer id (`seq.14148.2`), which is unreadable on a heatmap axis.
`feature_metadata.txt` maps each to a gene symbol.

**The map is not one-to-one**, several SOMAmers target the same protein. This record already
handles that: the publishers ship `GeneSymbol` pre-disambiguated, so the two ISG15 reagents arrive
as `ISG15_seq.14148.2` and `ISG15_seq.14151.4` rather than as two columns both called `ISG15`.
Verified against the original Zenodo archive, not just the unpacked copy.

The disambiguation below is therefore a no-op on this record, and it is kept anyway as a guard:
a record that did not pre-disambiguate would otherwise produce duplicate column names and silently
merge or drop reagents. The `stopifnot` is the part that matters.

In [4]:
fm  <- read.delim(raw("feature_metadata.txt"), check.names = FALSE)
sym <- setNames(fm$GeneSymbol, fm$SeqId)[colnames(L)]
sym[is.na(sym) | sym == ""] <- colnames(L)[is.na(sym) | sym == ""]
dup <- sym %in% sym[duplicated(sym)]
colnames(L) <- ifelse(dup, paste0(sym, "_", colnames(L)), sym)
stopifnot(!anyDuplicated(colnames(L)))

c(reagents = ncol(L), ambiguous_symbols = sum(dup),
  ISG15 = paste(grep("^ISG15", colnames(L), value = TRUE), collapse = " "))

reagents                     ambiguous_symbols 
                               "7288"                                   "0" 
                                ISG15 
"ISG15_seq.14148.2 ISG15_seq.14151.4"

## Age stays a band

The study releases `Age_group` in five-year bands and no exact age,, since exact age
is identifying. An earlier version replaced each band with its midpoint, which invents a measurement
that was never taken and buys nothing: the bands are equally spaced, so a band's index and its
midpoint differ by an affine transform and every correlation computed from them is numerically
identical.

Ordered by lower bound, not alphabetically. Alphabetical order happens to be correct for these
labels and would break silently on any band starting with a single digit.

In [5]:
lvl <- unique(meta$Age_group[!is.na(meta$Age_group) & meta$Age_group != ""])
lvl <- lvl[order(as.numeric(sub("-.*", "", lvl)))]
meta$Age_group <- factor(meta$Age_group, levels = lvl, ordered = TRUE)
levels(meta$Age_group)

[1] "16-20" "21-25" "26-30" "31-35" "36-40" "41-45" "46-50" "51-55" "56-60"
[10] "61-65" "66-70" "71-75" "76-80" "81-85"

## One sample per donor, keep the first visit

270 SLE samples come from 260 donors. Six donors were sampled repeatedly, and `SampleId` encodes
the visit: single-visit donors have a bare id (`D315`), repeat donors carry a suffix
(`D95_1 … D95_5`). Every correlation in weighted gene co-expression network analysis (WGCNA) assumes independent samples, so the network gets one
sample per donor.

**The first visit is also the most active disease**, which is why taking `_1` needs no separate
justification. SLEDAI-2K across the suffixes:

| donor | `_1` | `_2` | `_3` | `_4` | `_5` |
|---|---|---|---|---|---|
| D256 | 9 | 6 | 4 | 2 |, |
| D173 | 12 |, |, | 5 |, |
| D95 | 10 | 10 | 10 | 8 | 8 |

Activity falls with treatment over time, so "first by date" and "highest activity" pick the same
sample. The later visits are kept, written to their own matrix, and projected in the optional
step 09, where falling activity within one person becomes a test the cross-sectional analysis
cannot run.

In [6]:
visit <- suppressWarnings(as.integer(sub(".*_", "", meta$SampleId)))
visit[is.na(visit)] <- 1L                       # bare id = single visit
meta$Visit <- visit

first <- ave(visit, meta$DonorId, FUN = function(v) v == min(v)) == 1
repeats <- meta$Group == "SLE" & !first

c(total = nrow(meta),
  sle_first = sum(meta$Group == "SLE" & first),
  sle_later_visits = sum(repeats),
  healthy = sum(meta$Group == "HV"))

total        sle_first sle_later_visits          healthy 
             356              260               10               86

## The split

Three disjoint cohorts standing in for three institutions, drawn at random from the SLE samples and
stratified within `Batch` so each cohort carries batch A and B at the study proportion. A plain
shuffle gets that right on average and can miss on any single draw. With one sample per donor,
donor-disjointness is automatic.

They are not a batch split, batch is already corrected, and splitting on batch would make the
correction a no-op at every site.

**The assignment is frozen.** It is written once to `cohorts/cohort_assignment.csv` and read back on every
later run, so cohort membership never moves again and no downstream number changes because of a
reshuffle. Delete that file to redraw.

In [7]:
sle <- meta$Group == "SLE" & first
ASSIGN <- coh("cohort_assignment.csv")

if (file.exists(ASSIGN)) {
  a <- read.csv(ASSIGN, stringsAsFactors = FALSE)
  stopifnot(setequal(a$SampleId, meta$SampleId[sle]))
  cohort <- setNames(a$cohort, a$SampleId)[meta$SampleId[sle]]
  cat("cohort assignment read from", ASSIGN, "\n")
} else {
  set.seed(42)
  cohort <- character(sum(sle))
  b <- meta$Batch[sle]
  for (bb in sort(unique(b))) {
    i <- which(b == bb)
    cohort[sample(i)] <- rep_len(c("A","B","C"), length(i))
  }
  write.csv(data.frame(SampleId = meta$SampleId[sle],
                       DonorId  = meta$DonorId[sle], cohort = cohort),
            ASSIGN, row.names = FALSE)
  cat("cohort assignment drawn and frozen to", ASSIGN, "\n")
}
addmargins(table(cohort = cohort, batch = meta$Batch[sle]))

cohort assignment read from /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-proteomics/cohorts/cohort_assignment.csv 


,A,B,Sum
A,69,18,87
B,69,18,87
C,69,17,86
Sum,207,53,260


In [8]:
write_set <- function(rows, stem) {
  write.csv(L[rows, , drop = FALSE], coh("%s_log2_combat.csv", stem))
  write.csv(meta[rows, , drop = FALSE], coh("%s_meta.csv", stem))
  invisible(length(if (is.logical(rows)) which(rows) else rows))
}
ids <- meta$SampleId[sle]
for (ch in c("A","B","C")) write_set(ids[cohort == ch], sprintf("R_cohort-%s", ch))
write_set(meta$Group == "HV", "R_healthy")
write_set(repeats,            "R_timepoints")

stopifnot(!any(duplicated(meta$DonorId[sle])))
c(A = sum(cohort == "A"), B = sum(cohort == "B"), C = sum(cohort == "C"),
  healthy = sum(meta$Group == "HV"), later_visits = sum(repeats))

A            B            C      healthy later_visits 
          87           87           86           86           10

Five matrices, all on one corrected scale:

- `R_cohort-{A,B,C}`, the SLE network cohorts, one sample per donor
- `R_healthy`, the 86 healthy volunteers, held out of every fit and projected in step 08
- `R_timepoints`, later visits of the six repeat donors, projected in the optional step 09

All five land in `cohorts/`, beside `cohort_assignment.csv` and `clinical-traits.csv`, and are
committed. Nothing downstream touches the raw record again, from here on every notebook reads
`cohorts/` and writes `data/run_artifacts/`.

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [9]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:02 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
